# Classification Binaire de Tumeurs Cérébrales par IRM

**Équipe :** Redouane Hamecha · Malick Diatara · Yazid Aloui · Omar El Medjoubi · Mohamed Achoch · N'Guessan Yves Guichard Allou

---

Ce notebook présente une version finale, réorganisée et commentée du projet.

## 1. Imports et configuration

- bibliothèques
- seed
- détection du device

In [ ]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights, ResNet18_Weights

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Seed :", SEED)
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

## 2. Compréhension du problème

Le dataset original contient quatre classes.
Le problème est reformulé en classification binaire :

- 0 : absence de tumeur
- 1 : présence de tumeur

In [ ]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri"))
TRAIN_DIR = dataset_path / "Training"
TEST_DIR = dataset_path / "Testing"

IMG_EXTS = {".jpg", ".jpeg", ".png"}

def count_images(folder):
    counts = {}
    for cls_dir in sorted(Path(folder).iterdir()):
        if cls_dir.is_dir():
            n = len([f for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
            counts[cls_dir.name] = n
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("TRAIN_DIR :", TRAIN_DIR)
print("TEST_DIR  :", TEST_DIR)
print("Training :", train_counts)
print("Testing  :", test_counts)
print("Total train :", sum(train_counts.values()))
print("Total test  :", sum(test_counts.values()))

## 3. Analyse exploratoire

In [ ]:
tumor_count = sum(v for k, v in train_counts.items() if k != "no_tumor")
healthy_count = train_counts["no_tumor"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(
    list(train_counts.keys()),
    list(train_counts.values()),
    color=["#d9e8f5", "#b8d3ea", "#8fbce6", "#5c9ed6"]
)
axes[0].set_title("Distribution des classes originales")
axes[0].set_ylabel("Nombre d'images")
axes[0].tick_params(axis="x", rotation=20)

for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 10, f"{int(h)}", ha="center", fontsize=9)

bars = axes[1].bar(
    ["Pas de tumeur", "Tumeur"],
    [healthy_count, tumor_count],
    color=["#d9e8f5", "#5c9ed6"]
)
axes[1].set_title("Distribution binaire")
axes[1].set_ylabel("Nombre d'images")

for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 10, f"{int(h)}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

### Conclusion – Analyse exploratoire

- Le dataset est fortement déséquilibré
- Environ 86 % des images contiennent une tumeur contre 14 % sans tumeur
- Ce déséquilibre justifie l'utilisation du recall et du F1-score
- Les dimensions varient, ce qui impose un redimensionnement à 224 × 224

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
classes = sorted(train_counts.keys())

for j, cls_name in enumerate(classes):
    cls_dir = TRAIN_DIR / cls_name
    images = sorted([f for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS])[:2]
    for i, img_path in enumerate(images):
        img = Image.open(img_path).convert("RGB")
        axes[i, j].imshow(img)
        axes[i, j].set_title(cls_name)
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
widths, heights = [], []

for cls_dir in TRAIN_DIR.iterdir():
    if not cls_dir.is_dir():
        continue
    for img_path in list(cls_dir.iterdir())[:100]:
        if img_path.suffix.lower() in IMG_EXTS:
            img = Image.open(img_path)
            widths.append(img.size[0])
            heights.append(img.size[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=20, color="#d9e8f5", edgecolor="white")
axes[0].axvline(224, color="black", linestyle="--")
axes[0].set_title("Distribution des largeurs")

axes[1].hist(heights, bins=20, color="#8fbce6", edgecolor="white")
axes[1].axvline(224, color="black", linestyle="--")
axes[1].set_title("Distribution des hauteurs")

plt.tight_layout()
plt.show()

## 4. Préparation des données

In [ ]:
class BrainMRIDataset(Dataset):
    """Dataset binaire : 0 = sain, 1 = tumeur."""
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        for cls_dir in sorted(self.root_dir.iterdir()):
            if not cls_dir.is_dir():
                continue
            label = 0 if cls_dir.name == "no_tumor" else 1
            for img_path in sorted(cls_dir.iterdir()):
                if img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        label = torch.tensor(label, dtype=torch.long)
        return image, label

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224

def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_train_transforms_advanced():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=12, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.3),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_eval_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_tta_transforms():
    return [
        get_eval_transforms(),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.VerticalFlip(p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Rotate(limit=(10, 10), p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
    ]

In [ ]:
full_dataset = BrainMRIDataset(TRAIN_DIR, transform=None)
labels_all = [label for _, label in full_dataset.samples]
idx_all = np.arange(len(labels_all))

train_idx, val_idx = train_test_split(
    idx_all,
    test_size=0.2,
    random_state=SEED,
    stratify=labels_all,
)

train_dataset = BrainMRIDataset(TRAIN_DIR, transform=get_train_transforms())
val_dataset = BrainMRIDataset(TRAIN_DIR, transform=get_eval_transforms())
test_dataset = BrainMRIDataset(TEST_DIR, transform=get_eval_transforms())

train_set = Subset(train_dataset, train_idx)
val_set = Subset(val_dataset, val_idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train :", len(train_set))
print("Validation :", len(val_set))
print("Test :", len(test_dataset))

## 5. Fonctions communes

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    n_samples = 0
    correct = 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()

    return running_loss / n_samples, correct / n_samples


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n_samples = 0
    correct = 0
    all_targets, all_preds, all_probs = [], [], []

    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()

        all_targets.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    return {
        "loss": running_loss / n_samples,
        "acc": correct / n_samples,
        "targets": np.array(all_targets),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
    }


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred),
        "report": classification_report(y_true, y_pred, zero_division=0, digits=4),
    }


@torch.no_grad()
def predict_tta(model, dataset, device, tta_transforms=None):
    if tta_transforms is None:
        tta_transforms = get_tta_transforms()

    model.eval()
    all_probs, all_targets = [], []

    for idx in tqdm(range(len(dataset)), desc="TTA", leave=False):
        if hasattr(dataset, "indices"):
            img_path, label = dataset.dataset.samples[dataset.indices[idx]]
        else:
            img_path, label = dataset.samples[idx]

        image_np = np.array(Image.open(img_path).convert("RGB"))
        probs_per_version = []

        for t in tta_transforms:
            img_t = t(image=image_np)["image"].unsqueeze(0).to(device)
            output = model(img_t)
            prob = torch.softmax(output, dim=1)[0, 1].item()
            probs_per_version.append(prob)

        all_probs.append(np.mean(probs_per_version))
        all_targets.append(label)

    return np.array(all_targets), np.array(all_probs)


def predict_with_threshold(probs, threshold=0.5):
    return np.array([1 if p >= threshold else 0 for p in probs])


def evaluate_thresholds_with_tta(model, dataset, device, thresholds):
    y_true, probs = predict_tta(model, dataset, device)

    rows = []
    for thr in thresholds:
        preds = predict_with_threshold(probs, threshold=float(thr))
        rows.append([
            thr,
            precision_score(y_true, preds, zero_division=0),
            recall_score(y_true, preds, zero_division=0),
            f1_score(y_true, preds, zero_division=0),
        ])

    df = pd.DataFrame(rows, columns=["Seuil", "Precision", "Recall", "F1-score"])
    best_thr = df.loc[df["F1-score"].idxmax(), "Seuil"]
    best_preds = predict_with_threshold(probs, threshold=float(best_thr))
    metrics = compute_metrics(y_true, best_preds)

    return df, best_thr, metrics


def plot_training_curves(history, title_suffix=""):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
    axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation")
    axes[0].set_title(f"Loss {title_suffix}")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], marker="o", label="Train")
    axes[1].plot(epochs, history["val_acc"], marker="o", label="Validation")
    axes[1].set_title(f"Accuracy {title_suffix}")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def train_model_with_early_stopping(
    model,
    train_loader,
    val_loader,
    criterion_train,
    criterion_eval,
    optimizer,
    device,
    max_epochs=15,
    patience=3,
    checkpoint_path=None,
    label=""
):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    wait = 0

    for epoch in range(max_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion_train, optimizer, device)
        val_result = validate_one_epoch(model, val_loader, criterion_eval, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_result["loss"])
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_result["acc"])

        print(
            f"[{label}] Epoch {epoch+1:02d}/{max_epochs} | "
            f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | "
            f"Val loss: {val_result['loss']:.4f} | Val acc: {val_result['acc']:.4f}"
        )

        if val_result["loss"] < best_val_loss:
            best_val_loss = val_result["loss"]
            wait = 0
            if checkpoint_path is not None:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping déclenché pour {label}.")
                break

    return history

## 6. CNN from scratch et améliorations

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, img_size=224, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        flattened_dim = 128 * (img_size // 8) * (img_size // 8)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
save_dir = Path("artifacts/checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)

criterion_eval = nn.CrossEntropyLoss()

cnn_model_base = SmallCNN().to(DEVICE)
criterion_std = nn.CrossEntropyLoss()
optimizer_std = torch.optim.Adam(cnn_model_base.parameters(), lr=1e-3)

history_cnn_base = train_model_with_early_stopping(
    model=cnn_model_base,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_train=criterion_std,
    criterion_eval=criterion_eval,
    optimizer=optimizer_std,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "smallcnn_base_best.pt",
    label="CNN standard"
)

cnn_model_base.load_state_dict(torch.load(save_dir / "smallcnn_base_best.pt", map_location=DEVICE))
plot_training_curves(history_cnn_base, "(CNN standard)")

val_result = validate_one_epoch(cnn_model_base, val_loader, criterion_eval, DEVICE)
cnn_base_metrics = compute_metrics(val_result["targets"], val_result["preds"])

In [ ]:
cnn_model = SmallCNN().to(DEVICE)
criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)

history_cnn = train_model_with_early_stopping(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_train=criterion_smooth,
    criterion_eval=criterion_eval,
    optimizer=optimizer,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "smallcnn_best.pt",
    label="CNN loss ajustée"
)

cnn_model.load_state_dict(torch.load(save_dir / "smallcnn_best.pt", map_location=DEVICE))
plot_training_curves(history_cnn, "(CNN loss ajustée)")

val_result = validate_one_epoch(cnn_model, val_loader, criterion_eval, DEVICE)
cnn_metrics = compute_metrics(val_result["targets"], val_result["preds"])

In [ ]:
train_dataset_adv = BrainMRIDataset(TRAIN_DIR, transform=get_train_transforms_advanced())
train_set_adv = Subset(train_dataset_adv, train_idx)
train_loader_adv = DataLoader(train_set_adv, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

cnn_model_ft = SmallCNN().to(DEVICE)
cnn_model_ft.load_state_dict(torch.load(save_dir / "smallcnn_best.pt", map_location=DEVICE))
optimizer_cnn_ft = torch.optim.Adam(cnn_model_ft.parameters(), lr=5e-4)

history_cnn_ft = train_model_with_early_stopping(
    model=cnn_model_ft,
    train_loader=train_loader_adv,
    val_loader=val_loader,
    criterion_train=criterion_smooth,
    criterion_eval=criterion_eval,
    optimizer=optimizer_cnn_ft,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "smallcnn_ft_best.pt",
    label="CNN spécialisé"
)

cnn_model_ft.load_state_dict(torch.load(save_dir / "smallcnn_ft_best.pt", map_location=DEVICE))
plot_training_curves(history_cnn_ft, "(CNN spécialisé)")

val_result = validate_one_epoch(cnn_model_ft, val_loader, criterion_eval, DEVICE)
cnn_ft_metrics = compute_metrics(val_result["targets"], val_result["preds"])

thresholds = np.arange(0.30, 0.71, 0.05)
cnn_threshold_df, best_threshold_cnn, cnn_tta_metrics = evaluate_thresholds_with_tta(
    cnn_model_ft, val_set, DEVICE, thresholds
)

cnn_progress_df = pd.DataFrame({
    "Version CNN": ["Base", "Loss ajustée", "Spécialisé", "Spécialisé + TTA"],
    "Accuracy": [cnn_base_metrics["accuracy"], cnn_metrics["accuracy"], cnn_ft_metrics["accuracy"], cnn_tta_metrics["accuracy"]],
    "Precision": [cnn_base_metrics["precision"], cnn_metrics["precision"], cnn_ft_metrics["precision"], cnn_tta_metrics["precision"]],
    "Recall": [cnn_base_metrics["recall"], cnn_metrics["recall"], cnn_ft_metrics["recall"], cnn_tta_metrics["recall"]],
    "F1-score": [cnn_base_metrics["f1"], cnn_metrics["f1"], cnn_ft_metrics["f1"], cnn_tta_metrics["f1"]],
})

cnn_progress_df

### Conclusion – Modèles CNN

- Le CNN standard atteint déjà de bonnes performances
- Le label smoothing n'apporte pas d'amélioration nette
- Le fine-tuning du CNN améliore surtout la robustesse
- Le TTA apporte un gain modéré
- Le CNN reste en dessous des modèles pré-entraînés

## 7. ResNet18 pré-entraîné

In [ ]:
def build_resnet18(num_classes=2, pretrained=True):
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

def freeze_backbone_resnet(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True
    return model

def unfreeze_last_blocks_resnet(model):
    for p in model.parameters():
        p.requires_grad = False
    for layer_name in ["layer3", "layer4"]:
        for p in getattr(model, layer_name).parameters():
            p.requires_grad = True
    for p in model.fc.parameters():
        p.requires_grad = True
    return model

In [ ]:
resnet_model = build_resnet18(num_classes=2, pretrained=True).to(DEVICE)
resnet_model = freeze_backbone_resnet(resnet_model)

criterion_resnet = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer_resnet_a = torch.optim.Adam(resnet_model.fc.parameters(), lr=1e-3)

history_resnet_a = train_model_with_early_stopping(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_train=criterion_resnet,
    criterion_eval=criterion_eval,
    optimizer=optimizer_resnet_a,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "resnet18_phase_a.pt",
    label="ResNet18 phase A"
)

resnet_model.load_state_dict(torch.load(save_dir / "resnet18_phase_a.pt", map_location=DEVICE))

In [ ]:
resnet_model = build_resnet18(num_classes=2, pretrained=False).to(DEVICE)
resnet_model.load_state_dict(torch.load(save_dir / "resnet18_phase_a.pt", map_location=DEVICE))
resnet_model = unfreeze_last_blocks_resnet(resnet_model)

optimizer_resnet_b = torch.optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=1e-4)

history_resnet_b = train_model_with_early_stopping(
    model=resnet_model,
    train_loader=train_loader_adv,
    val_loader=val_loader,
    criterion_train=criterion_resnet,
    criterion_eval=criterion_eval,
    optimizer=optimizer_resnet_b,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "resnet18_best.pt",
    label="ResNet18 phase B"
)

resnet_model.load_state_dict(torch.load(save_dir / "resnet18_best.pt", map_location=DEVICE))

history_resnet = {
    "train_loss": history_resnet_a["train_loss"] + history_resnet_b["train_loss"],
    "val_loss": history_resnet_a["val_loss"] + history_resnet_b["val_loss"],
    "train_acc": history_resnet_a["train_acc"] + history_resnet_b["train_acc"],
    "val_acc": history_resnet_a["val_acc"] + history_resnet_b["val_acc"],
}

plot_training_curves(history_resnet, "(ResNet18)")

val_result = validate_one_epoch(resnet_model, val_loader, criterion_eval, DEVICE)
resnet_metrics = compute_metrics(val_result["targets"], val_result["preds"])
resnet_threshold_df, best_threshold_resnet, resnet_tta_metrics = evaluate_thresholds_with_tta(
    resnet_model, val_set, DEVICE, thresholds
)

### Conclusion – ResNet18

- Le transfer learning apporte un gain significatif
- Le fine-tuning des derniers blocs améliore fortement les résultats
- Les scores de validation sont très élevés
- La généralisation doit être confirmée sur le jeu de test

## 8. EfficientNet-B0 pré-entraîné

In [ ]:
def build_efficientnet(num_classes=2, pretrained=True):
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

def freeze_backbone_efficientnet(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    return model

def unfreeze_last_blocks_efficientnet(model):
    for p in model.parameters():
        p.requires_grad = False
    for idx in [6, 7, 8]:
        for p in model.features[idx].parameters():
            p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True
    return model

In [ ]:
eff_model = build_efficientnet(num_classes=2, pretrained=True).to(DEVICE)
eff_model = freeze_backbone_efficientnet(eff_model)

criterion_eff = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer_eff_a = torch.optim.Adam(eff_model.classifier.parameters(), lr=1e-3)

history_eff_a = train_model_with_early_stopping(
    model=eff_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_train=criterion_eff,
    criterion_eval=criterion_eval,
    optimizer=optimizer_eff_a,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "efficientnet_phase_a.pt",
    label="EfficientNet phase A"
)

eff_model.load_state_dict(torch.load(save_dir / "efficientnet_phase_a.pt", map_location=DEVICE))

In [ ]:
eff_model = build_efficientnet(num_classes=2, pretrained=False).to(DEVICE)
eff_model.load_state_dict(torch.load(save_dir / "efficientnet_phase_a.pt", map_location=DEVICE))
eff_model = unfreeze_last_blocks_efficientnet(eff_model)

optimizer_eff_b = torch.optim.Adam(filter(lambda p: p.requires_grad, eff_model.parameters()), lr=1e-4)

history_eff_b = train_model_with_early_stopping(
    model=eff_model,
    train_loader=train_loader_adv,
    val_loader=val_loader,
    criterion_train=criterion_eff,
    criterion_eval=criterion_eval,
    optimizer=optimizer_eff_b,
    device=DEVICE,
    max_epochs=15,
    patience=3,
    checkpoint_path=save_dir / "efficientnet_best.pt",
    label="EfficientNet phase B"
)

eff_model.load_state_dict(torch.load(save_dir / "efficientnet_best.pt", map_location=DEVICE))

history_eff = {
    "train_loss": history_eff_a["train_loss"] + history_eff_b["train_loss"],
    "val_loss": history_eff_a["val_loss"] + history_eff_b["val_loss"],
    "train_acc": history_eff_a["train_acc"] + history_eff_b["train_acc"],
    "val_acc": history_eff_a["val_acc"] + history_eff_b["val_acc"],
}

plot_training_curves(history_eff, "(EfficientNet-B0)")

val_result = validate_one_epoch(eff_model, val_loader, criterion_eval, DEVICE)
eff_metrics = compute_metrics(val_result["targets"], val_result["preds"])
eff_threshold_df, best_threshold_eff, eff_tta_metrics = evaluate_thresholds_with_tta(
    eff_model, val_set, DEVICE, thresholds
)

### Conclusion – EfficientNet-B0

- EfficientNet-B0 donne des performances élevées et plus stables
- La loss de validation reste faible après fine-tuning
- Le modèle semble mieux généraliser que ResNet18
- Il est retenu comme meilleur modèle final

## 9. Comparaison sur validation

In [ ]:
val_comparison_df = pd.DataFrame({
    "Modèle": [
        "CNN spécialisé + TTA",
        "ResNet18 + TTA",
        "EfficientNet-B0",
        "EfficientNet-B0 + TTA",
    ],
    "Accuracy": [
        cnn_tta_metrics["accuracy"],
        resnet_tta_metrics["accuracy"],
        eff_metrics["accuracy"],
        eff_tta_metrics["accuracy"],
    ],
    "Precision": [
        cnn_tta_metrics["precision"],
        resnet_tta_metrics["precision"],
        eff_metrics["precision"],
        eff_tta_metrics["precision"],
    ],
    "Recall": [
        cnn_tta_metrics["recall"],
        resnet_tta_metrics["recall"],
        eff_metrics["recall"],
        eff_tta_metrics["recall"],
    ],
    "F1-score": [
        cnn_tta_metrics["f1"],
        resnet_tta_metrics["f1"],
        eff_metrics["f1"],
        eff_tta_metrics["f1"],
    ],
})

val_comparison_df.sort_values("F1-score", ascending=False)

### Conclusion – Comparaison sur validation

- Les modèles pré-entraînés surpassent le CNN
- ResNet18 et EfficientNet-B0 sont les deux meilleurs candidats
- EfficientNet-B0 est retenu pour la phase finale en raison de sa stabilité

## 10. Optimisation du seuil pour EfficientNet-B0

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.02)

y_true_eff_val, probs_eff_val = predict_tta(eff_model, val_set, DEVICE)

rows = []
for thr in thresholds:
    preds_thr = predict_with_threshold(probs_eff_val, threshold=float(thr))
    rows.append({
        "threshold": round(float(thr), 2),
        "precision": precision_score(y_true_eff_val, preds_thr, zero_division=0),
        "recall": recall_score(y_true_eff_val, preds_thr, zero_division=0),
        "f1": f1_score(y_true_eff_val, preds_thr, zero_division=0),
    })

eff_threshold_analysis = pd.DataFrame(rows)
eff_threshold_analysis.head()

In [ ]:
best_threshold_f1 = eff_threshold_analysis.loc[
    eff_threshold_analysis["f1"].idxmax(), "threshold"
]

FINAL_THRESHOLD = float(best_threshold_f1)
print("Seuil final retenu :", FINAL_THRESHOLD)

In [ ]:
plt.hist(probs_eff_val, bins=30)
plt.title("Distribution des probabilités (EfficientNet)")
plt.xlabel("Probabilité tumeur")
plt.ylabel("Nombre d'images")
plt.show()

### Conclusion – Optimisation du seuil

- La séparation des probabilités est très nette
- Les images saines sont proches de 0 et les tumeurs proches de 1
- Le choix du seuil a donc un impact limité autour de la zone optimale
- Le seuil retenu est 0.18 car il maximise le F1-score sur validation

## 11. Évaluation finale sur le jeu de test

In [ ]:
test_results = {}

for name, model_used, thr in [
    ("CNN spécialisé + TTA", cnn_model_ft, 0.5),
    ("ResNet18 + TTA", resnet_model, 0.5),
    ("EfficientNet-B0 + TTA", eff_model, FINAL_THRESHOLD),
]:
    y_true_test, probs_test = predict_tta(model_used, test_dataset, DEVICE)
    preds_test = predict_with_threshold(probs_test, threshold=thr)
    metrics_test = compute_metrics(y_true_test, preds_test)

    test_results[name] = {
        "y_true": y_true_test,
        "probs": probs_test,
        "preds": preds_test,
        "metrics": metrics_test,
    }

test_summary_df = pd.DataFrame({
    "Modèle": list(test_results.keys()),
    "Accuracy": [test_results[k]["metrics"]["accuracy"] for k in test_results],
    "Precision": [test_results[k]["metrics"]["precision"] for k in test_results],
    "Recall": [test_results[k]["metrics"]["recall"] for k in test_results],
    "F1-score": [test_results[k]["metrics"]["f1"] for k in test_results],
})

test_summary_df

In [ ]:
model_names = list(test_results.keys())
recall_vals = [test_results[k]["metrics"]["recall"] * 100 for k in model_names]
f1_vals = [test_results[k]["metrics"]["f1"] * 100 for k in model_names]

x = np.arange(len(model_names))
width = 0.32

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, recall_vals, width, label="Recall (%)", color="#8fbce6")
bars2 = ax.bar(x + width/2, f1_vals, width, label="F1-score (%)", color="#5c9ed6")

ax.set_title("Comparaison des méthodes sur le jeu de test")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.set_ylim(0, 100)
ax.legend()

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, h + 0.8, f"{h:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

### Conclusion – Jeu de test

- Les modèles pré-entraînés confirment leur supériorité
- ResNet18 reste très performant
- EfficientNet-B0 + TTA avec seuil optimisé offre le meilleur compromis global
- Le modèle final retenu est donc EfficientNet-B0

## 12. Sauvegarde du modèle final

In [ ]:
best_model_path = save_dir / "best_efficientnet_b0_final.pt"
torch.save(eff_model.state_dict(), best_model_path)
print("Modèle sauvegardé dans :", best_model_path)

## 13. Interprétabilité avec Grad-CAM

In [ ]:
!pip install grad-cam --quiet

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def generate_gradcam(model, image_tensor, image_np, target_layers, class_idx=None):
    model.eval()

    with torch.no_grad():
        outputs = model(image_tensor)
        pred_class = torch.argmax(outputs, dim=1).item()

    if class_idx is None:
        class_idx = pred_class

    targets = [ClassifierOutputTarget(class_idx)]

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=image_tensor, targets=targets)[0]

    h, w = grayscale_cam.shape
    image_resized = np.array(Image.fromarray(image_np).resize((w, h))).astype(np.float32) / 255.0
    cam_overlay = show_cam_on_image(image_resized, grayscale_cam, use_rgb=True)

    return pred_class, grayscale_cam, cam_overlay, image_resized

def label_to_text(label):
    return "Tumeur" if label == 1 else "Sain"

def label_to_binary(label):
    return "Sain" if label == "no_tumor" else "Tumeur"

In [ ]:
import random

image_paths = []
for label in os.listdir(TEST_DIR):
    folder = os.path.join(TEST_DIR, label)
    for img in os.listdir(folder):
        image_paths.append((os.path.join(folder, img), label))

healthy = [x for x in image_paths if x[1] == "no_tumor"]
tumor = [x for x in image_paths if x[1] != "no_tumor"]

samples = [
    random.choice(healthy),
    random.choice(tumor),
    random.choice(tumor),
]

target_layer_eff = [eff_model.features[-1]]

fig, axes = plt.subplots(3, 3, figsize=(14, 12))

for i, (img_path, label) in enumerate(samples):
    pil_img = Image.open(img_path).convert("RGB")
    image_np = np.array(pil_img)
    image_tensor = get_eval_transforms()(image=image_np)["image"].unsqueeze(0).to(DEVICE)

    pred_eff, cam_eff, overlay_eff, image_resized = generate_gradcam(
        model=eff_model,
        image_tensor=image_tensor,
        image_np=image_np,
        target_layers=target_layer_eff,
        class_idx=None
    )

    axes[i, 0].imshow(image_resized)
    axes[i, 0].set_title(f"Vrai : {label_to_binary(label)}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cam_eff, cmap="jet")
    axes[i, 1].set_title("Grad-CAM")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(overlay_eff)
    axes[i, 2].set_title(f"Prédiction : {label_to_text(pred_eff)}")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

### Conclusion – Grad-CAM

- Grad-CAM met en évidence les zones importantes pour la prédiction
- Le modèle final se concentre globalement sur des régions plausibles
- Cette analyse complète utilement les métriques quantitatives

## 14. Conclusion générale

- Le transfer learning améliore nettement la classification des IRM cérébrales
- Les CNN from scratch restent utiles comme baseline
- ResNet18 et EfficientNet-B0 apportent un gain important
- EfficientNet-B0 offre le meilleur compromis final
- Le TTA et l'optimisation du seuil renforcent la performance globale
- Grad-CAM améliore l'interprétabilité du système

### Limites

- dataset déséquilibré
- nombre d'images limité
- absence de validation clinique externe

### Perspectives

- calibration des probabilités
- ajout de données
- exploration d'architectures plus avancées